# SVM with Linear Kernel — Augmented Dataset (500 per class)
## Wisconsin Breast Cancer Dataset
### Pipeline: Load → 80/20 Split → Scale → Train SVM(linear) → Blue Visualizations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve

sns.set_palette('Blues')
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.Blues(np.linspace(0.3, 0.9, 8)))

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load augmented dataset (500 per class)
df = pd.read_csv('../../augmented_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Class distribution
print(f"Class distribution:\n{df['diagnosis'].value_counts()}")
print(f"\nMalignant (1): {df['diagnosis'].sum()} cases")
print(f"Benign    (0): {len(df) - df['diagnosis'].sum()} cases")

In [ ]:
# Features and target
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {X_train.shape[0]} samples")
print(f"Test size:  {X_test.shape[0]} samples")
print(f"\nTrain distribution:\n{y_train.value_counts()}")
print(f"Test distribution:\n{y_test.value_counts()}")

In [ ]:
# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features standardized.")

In [ ]:
# Train SVM Linear
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)

y_pred = svm_model.predict(X_test_scaled)
y_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

print("SVM Linear trained on augmented data.")

In [ ]:
# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
roc_auc = roc_auc_score(y_test, y_proba)

print("=" * 50)
print("SVM Linear Kernel — Augmented Data Results")
print("=" * 50)
print(f"Accuracy:     {accuracy:.4f}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"F1-score:     {f1:.4f}")
print(f"Specificity:  {specificity:.4f}")
print(f"ROC-AUC:      {roc_auc:.4f}")
print("\nConfusion Matrix:")
print(f"              Predicted")
print(f"              Neg    Pos")
print(f"Actual Neg    {tn:3d}   {fp:2d}")
print(f"       Pos    {fn:2d}   {tp:3d}")

In [ ]:
# Blue confusion matrix heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'],
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix — SVM Linear (Augmented Data)', fontsize=13, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Blue metrics bar chart
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'Specificity']
metrics_values = [accuracy, precision, recall, f1, specificity]
bar_colors = plt.cm.Blues([0.4, 0.5, 0.6, 0.7, 0.8])

plt.figure(figsize=(9, 5))
bars = plt.bar(metrics_names, metrics_values, color=bar_colors, edgecolor='darkblue', linewidth=1.2)
plt.ylim(0, 1.05)
plt.title('Model Performance Metrics — SVM Linear (Augmented Data)', fontsize=13, fontweight='bold')
plt.ylabel('Score')

for bar, val in zip(bars, metrics_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Blue ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='royalblue', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.fill_between(fpr, tpr, alpha=0.15, color='royalblue')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', alpha=0.7)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — SVM Linear (Augmented Data)', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()